# 02 - Data Cleaning & Standardization
Runs the reusable cleaning/standardization functions from `src/` on each raw dataset and shows the raw-vs-cleaned row count proof required for Gate 1.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
from src import ingestion, cleaning, validation, feature_engineering as fe

## Load raw data

In [2]:
raw = ingestion.load_all_raw()
raw_counts = {name: len(df) for name, df in raw.items()}

## Apply cleaning + standardization

In [3]:
cleaned = {
    'mandi_master': cleaning.clean_mandi_master(raw['mandi_master']),
    'arrivals': cleaning.clean_arrivals(raw['arrivals']),
    'prices': cleaning.clean_prices(raw['prices']),
    'msp': cleaning.clean_msp(raw['msp']),
    'weather': cleaning.clean_weather(raw['weather']),
    'transport': cleaning.clean_transport(raw['transport']),
}
cleaned_counts = {name: len(df) for name, df in cleaned.items()}

## RAW vs CLEANED row counts (Gate 1 requirement)

In [4]:
summary = pd.DataFrame({'raw_rows': raw_counts, 'cleaned_rows': cleaned_counts})
summary['rows_removed'] = summary['raw_rows'] - summary['cleaned_rows']
summary.loc['TOTAL'] = summary.sum()
summary

,raw_rows,cleaned_rows,rows_removed
mandi_master,60,57,3
arrivals,12180,12000,180
prices,9000,9000,0
msp,7,7,0
weather,6000,6000,0
transport,8000,8000,0
TOTAL,35247,35064,183


## Spot-check: crop normalization worked

In [5]:
cleaned['arrivals'][['crop_name', 'crop']].drop_duplicates().sample(10, random_state=1)

,crop_name,crop
18,Makka,Maize
26,मक्का,Maize
3,धान,Rice
59,गेहूँ,Wheat
167,wheat,Wheat
45,Sugarcane,Sugarcane
27,Ganna,Sugarcane
44,maize,Maize
40,Corn,Maize
101,rice,Rice


## Spot-check: mandi_id normalization worked

In [6]:
raw['arrivals']['mandi_id'].head(5).tolist(), cleaned['arrivals']['mandi_id'].head(5).tolist()

(['036', '043', '016', 'MANDI-045', 'mandi055'],
 ['MANDI036', 'MANDI043', 'MANDI016', 'MANDI045', 'MANDI055'])

## Spot-check: unit conversion to Quintal

In [7]:
cleaned['arrivals'][['arrival_quantity', 'unit', 'quantity_quintal']].head(10)

,arrival_quantity,unit,quantity_quintal
0,147.926,MT,1479.2600
1,78.393,Kilo,0.7839
2,172.033,kg,1.7203
3,412.828,Q,412.8280
4,71.225,Tonnes,712.2500
5,71.226,QTL,71.2260
6,436.685,qtl,436.6850
7,193.918,Kg,1.9392
8,NaN,MT,NaN
9,154.866,Kilo,1.5487


## Validation checks (referential integrity, dates, negative values)

In [8]:
validation_results = validation.run_all_validations(cleaned)
pd.DataFrame(validation_results)

,child_dataset,parent_dataset,total_child_rows,matched_rows,unmatched_rows,match_rate_pct,dataset,column,invalid_dates_remaining,total_rows,remaining_negative_count,out_of_range_count
0,arrivals,mandi_master,12000.0,12000.0,0.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN
1,transport,mandi_master,8000.0,8000.0,0.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,arrivals,arrival_date,0.0,12000.0,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,prices,price_date,0.0,9000.0,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,arrivals,arrival_quantity,NaN,NaN,0.0,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,weather,rainfall_mm,NaN,NaN,0.0,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,transport,transit_hours_clean,NaN,NaN,0.0,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,weather,temperature_c,NaN,6000.0,NaN,221.0
8,NaN,NaN,NaN,NaN,NaN,NaN,weather,humidity_percent,NaN,6000.0,NaN,0.0


## Feature engineering (joins + derived KPIs feed columns)

In [9]:
arrivals_feat = fe.build_arrivals_features(cleaned['arrivals'], cleaned['mandi_master'])
price_feat = fe.build_price_features(cleaned['prices'], cleaned['msp'])
weather_daily = fe.build_weather_daily(cleaned['weather'])
market_value = fe.build_market_value(arrivals_feat, price_feat)
arrivals_feat.shape, price_feat.shape, weather_daily.shape, market_value.shape

((11652, 15), (9000, 17), (289, 5), (13147, 21))

This is the same logic executed end-to-end and reproducibly by `python -m src.pipeline`, which also writes the cleaned/feature tables to `data/processed/` and the SQLite database.